In [1]:
LAST_NAME = "SANTARIN"
FAVORITE_ARTIST = "UNCLE DAGS"
STUDENT_ID = "TUPM-26-1865"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)
ARTIST_LENGTH = len(FAVORITE_ARTIST)
ARTIST_SUM = sum(ord(c) - ord("A") + 1 for c in FAVORITE_ARTIST.upper() if c.isalpha())

print("Seed Digit   :", SEED_DIGIT)
print("ID Sum       :", ID_SUM)
print("Name Length  :", NAME_LENGTH)
print("Artist Length:", ARTIST_LENGTH)
print("Artist Sum   :", ARTIST_SUM)

Seed Digit   : 5
ID Sum       : 28
Name Length  : 8
Artist Length: 10
Artist Sum   : 86


In [2]:
def linear_search(data, target):
    comparisons = 0
    for i, value in enumerate(data):
        comparisons += 1
        if value == target:
            return i, comparisons
    return -1, comparisons


# Data generation rules
N_RECORDS = 20
incident_records = [1000 + i * (SEED_DIGIT + 1) for i in range(N_RECORDS)]   # unique, ascending

rotation = SEED_DIGIT % N_RECORDS                                            # rotation point
rotated_records = incident_records[rotation:] + incident_records[:rotation]  # first k records moved to the end

target_incident = incident_records[(ID_SUM + ARTIST_SUM) % N_RECORDS]        # target from ID_SUM and ARTIST_SUM

index, comparisons = linear_search(rotated_records, target_incident)

print("Original Dataset :", incident_records)
print("Rotation Point   :", rotation)
print("Rotated Dataset  :", rotated_records)
print("Target Incident  :", target_incident)
if index != -1:
    print(f"Result           : FOUND at index {index} (position {index + 1})")
else:
    print("Result           : NOT FOUND")
print("Comparison Count :", comparisons)

# Not-found case (worst case)
missing = 9999
m_index, m_comparisons = linear_search(rotated_records, missing)
print(f"\nTarget {missing}: index={m_index} (not found), comparisons={m_comparisons}")

print("\nTime Complexity : O(n) - linear search checks records one by one")
print("Space Complexity: O(1) - only counters and index variables are used")

Original Dataset : [1000, 1006, 1012, 1018, 1024, 1030, 1036, 1042, 1048, 1054, 1060, 1066, 1072, 1078, 1084, 1090, 1096, 1102, 1108, 1114]
Rotation Point   : 5
Rotated Dataset  : [1030, 1036, 1042, 1048, 1054, 1060, 1066, 1072, 1078, 1084, 1090, 1096, 1102, 1108, 1114, 1000, 1006, 1012, 1018, 1024]
Target Incident  : 1084
Result           : FOUND at index 9 (position 10)
Comparison Count : 10

Target 9999: index=-1 (not found), comparisons=20

Time Complexity : O(n) - linear search checks records one by one
Space Complexity: O(1) - only counters and index variables are used


In [3]:
def merge_sorted(a, b):
    merged, i, j, comparisons = [], 0, 0, 0
    while i < len(a) and j < len(b):
        comparisons += 1
        if a[i] <= b[j]:
            merged.append(a[i])
            i += 1
        else:
            merged.append(b[j])
            j += 1
    merged.extend(a[i:])   # leftovers need no comparisons
    merged.extend(b[j:])
    return merged, comparisons


# Data generation rules (both strictly ascending)
dataset_a = [ID_SUM + i * SEED_DIGIT for i in range(10)]           # uses SEED_DIGIT and ID_SUM
dataset_b = [ARTIST_LENGTH + i * NAME_LENGTH for i in range(12)]   # uses ARTIST_LENGTH and NAME_LENGTH

consolidated, comparisons = merge_sorted(dataset_a, dataset_b)

print("Department A Dataset  :", dataset_a)
print("Department B Dataset  :", dataset_b)
print("Consolidated Dataset  :", consolidated)
print("Records Before        :", len(dataset_a), "+", len(dataset_b), "=", len(dataset_a) + len(dataset_b))
print("Records After         :", len(consolidated))
print("Comparison Count      :", comparisons)
print("Ordering Verification :", consolidated == sorted(dataset_a + dataset_b)
      and all(consolidated[k] <= consolidated[k + 1] for k in range(len(consolidated) - 1)))

print("\nTime Complexity : O(n + m) - each record is examined once")
print("Space Complexity: O(n + m) - a new consolidated list is created")

Department A Dataset  : [28, 33, 38, 43, 48, 53, 58, 63, 68, 73]
Department B Dataset  : [10, 18, 26, 34, 42, 50, 58, 66, 74, 82, 90, 98]
Consolidated Dataset  : [10, 18, 26, 28, 33, 34, 38, 42, 43, 48, 50, 53, 58, 58, 63, 66, 68, 73, 74, 82, 90, 98]
Records Before        : 10 + 12 = 22
Records After         : 22
Comparison Count      : 18
Ordering Verification : True

Time Complexity : O(n + m) - each record is examined once
Space Complexity: O(n + m) - a new consolidated list is created


In [4]:
import time


def bubble_sort(data):
    a, comparisons, swaps = data.copy(), 0, 0
    for i in range(len(a)):
        swapped = False
        for j in range(len(a) - i - 1):
            comparisons += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swaps += 1
                swapped = True
        if not swapped:
            break
    return a, comparisons, swaps


def selection_sort(data):
    a, comparisons, swaps = data.copy(), 0, 0
    for i in range(len(a) - 1):
        minimum = i
        for j in range(i + 1, len(a)):
            comparisons += 1
            if a[j] < a[minimum]:
                minimum = j
        if minimum != i:
            a[i], a[minimum] = a[minimum], a[i]
            swaps += 1
    return a, comparisons, swaps


def insertion_sort(data):
    a, comparisons, shifts = data.copy(), 0, 0
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0:
            comparisons += 1
            if a[j] <= key:
                break
            a[j + 1] = a[j]
            shifts += 1
            j -= 1
        a[j + 1] = key
    return a, comparisons, shifts


def measure(sort_function, data, trials=5):
    times = []
    for _ in range(trials):
        start = time.perf_counter()
        sort_function(data)
        times.append(time.perf_counter() - start)
    return sum(times) / trials


# Data generation (unique order numbers; 1009 is prime so values never repeat for N < 1009)
N_ORDERS = 25
unsorted_orders = [1000 + (ID_SUM * ARTIST_SUM * i + SEED_DIGIT) % 1009 for i in range(N_ORDERS)]
reverse_orders = sorted(unsorted_orders, reverse=True)
sorted_orders = sorted(unsorted_orders)

conditions = [
    ("Unsorted", unsorted_orders),
    ("Reverse", reverse_orders),
    ("Already Sorted", sorted_orders),
]
algorithms = [
    ("Bubble Sort", bubble_sort),
    ("Selection Sort", selection_sort),
    ("Insertion Sort", insertion_sort),
]

for cond_name, orders in conditions:
    print(f"\n=== {cond_name} Orders ===")
    print("Input :", orders)
    print(f"{'Algorithm':<16}{'Comparisons':>12}{'Moves':>8}{'Time (s)':>14}  Verified")
    for algo_name, func in algorithms:
        result, comps, moves = func(orders)
        elapsed = measure(func, orders)
        ok = result == sorted(orders)
        print(f"{algo_name:<16}{comps:>12}{moves:>8}{elapsed:>14.8f}  {ok}")

print("\nTime Complexity : Bubble O(n^2) | Selection O(n^2) | Insertion O(n^2) worst, O(n) best")
print("Space Complexity: O(1) extra space for all three (sorting a working copy)")


=== Unsorted Orders ===
Input : [1005, 1395, 1785, 1166, 1556, 1946, 1327, 1717, 1098, 1488, 1878, 1259, 1649, 1030, 1420, 1810, 1191, 1581, 1971, 1352, 1742, 1123, 1513, 1903, 1284]
Algorithm        Comparisons   Moves      Time (s)  Verified
Bubble Sort              285     136    0.00002614  True
Selection Sort           300      22    0.00001692  True
Insertion Sort           160     136    0.00001368  True

=== Reverse Orders ===
Input : [1971, 1946, 1903, 1878, 1810, 1785, 1742, 1717, 1649, 1581, 1556, 1513, 1488, 1420, 1395, 1352, 1327, 1284, 1259, 1191, 1166, 1123, 1098, 1030, 1005]
Algorithm        Comparisons   Moves      Time (s)  Verified
Bubble Sort              300     300    0.00003652  True
Selection Sort           300      12    0.00001644  True
Insertion Sort           300     300    0.00002710  True

=== Already Sorted Orders ===
Input : [1005, 1030, 1098, 1123, 1166, 1191, 1259, 1284, 1327, 1352, 1395, 1420, 1488, 1513, 1556, 1581, 1649, 1717, 1742, 1785, 1810, 187